# Gauthier_Cerema : Test Steger & Frangi

Ce notebook implémente les premiers tests demandés :
1. Téléchargement des données via `gdown`.
2. Téléchargement ciblé du script `steger_gpu.py` depuis GitHub (pas de clonage complet).
3. Filtre Frangi classique (skimage) avec $\sigma=30$.
4. Calcul du Hessien Gaussien (implémentation GPU via PyTorch) avec $\sigma=30$ sur l'image spécifique 'Ombrage 05m base.tif'.

In [ ]:
!pip install rasterio torch torchvision gdown

# 1. Téléchargement du fichier nécessaire depuis GitHub
steger_url = "https://raw.githubusercontent.com/Ludwig-H/Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/main/Gauthier_Cerema/steger_gpu.py"
!wget {steger_url} -O steger_gpu.py

# 2. Téléchargement des données (gdown)
# ID du dossier Drive : 1lWi-7oZ064QSh_NjX_xrA5iv03A2hYjQ
!gdown --folder https://drive.google.com/drive/folders/1lWi-7oZ064QSh_NjX_xrA5iv03A2hYjQ -O Gauthier_Cerema_Data

In [ ]:
import sys
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from skimage.filters import frangi
import torch

# Import depuis le fichier téléchargé localement
try:
    from steger_gpu import StegerHessian
    print("Module StegerHessian importé avec succès.")
except ImportError as e:
    print(f"Erreur d'import : {e}. Vérifiez que steger_gpu.py a bien été téléchargé.")

In [ ]:
# --- DEFINITION DU CHEMIN ---
# Les images sont dans le sous-dossier 'Premier_test' du téléchargement
base_data_path = 'Gauthier_Cerema_Data'
folder_path = os.path.join(base_data_path, 'Premier_test')

if os.path.exists(folder_path):
    print(f"Dossier trouvé : {folder_path}")
    files = os.listdir(folder_path)
    tif_files = [f for f in files if f.lower().endswith('.tif')]
    print(f"Fichiers .tif trouvés ({len(tif_files)}) :", tif_files)
    
    # --- Affichage des images brutes ---
    for f in tif_files:
        f_path = os.path.join(folder_path, f)
        try:
            with rasterio.open(f_path) as src:
                img = src.read(1)
                plt.figure(figsize=(6, 6))
                plt.imshow(img, cmap='gray')
                plt.title(f"Image brute : {f}")
                plt.axis('off')
                plt.show()
        except Exception as e:
            print(f"Impossible de lire {f} : {e}")
else:
    print(f"ATTENTION : Le dossier {folder_path} n'a pas été trouvé.")
    if os.path.exists(base_data_path):
        print(f"Contenu de {base_data_path} :", os.listdir(base_data_path))

In [ ]:
# --- 1. FILTRE FRANGI CLASSIQUE (tous les tifs) ---
σ_val = 30

if os.path.exists(folder_path):
    tif_files = [f for f in os.listdir(folder_path) if f.endswith('.tif')]

    for file_name in tif_files:
        file_path = os.path.join(folder_path, file_name)
        try:
            with rasterio.open(file_path) as src:
                data = src.read(1)
                
                # Frangi Classique
                print(f"Traitement Frangi (σ={σ_val}) sur {file_name}...")
                response = frangi(data, sigmas=[σ_val])

                plt.figure(figsize=(10, 6))
                plt.imshow(response, cmap='gray')
                plt.colorbar(label='Frangi Response')
                plt.title(f"Frangi (σ={σ_val}) : {file_name}")
                plt.show()
        except Exception as e:
            print(f"Erreur lecture {file_name}: {e}")

In [ ]:
# --- 2. HESSIEN GAUSSIEN (Cibles : Ombrage et MNT) ---
target_files = ["Ombrage 05m base.tif", "MNT 05.tif"]

for target_file in target_files:
    target_path = os.path.join(folder_path, target_file)

    if os.path.exists(target_path):
        print(f"Calcul du Hessien Gaussien sur {target_file}...")
        
        try:
            with rasterio.open(target_path) as src:
                data = src.read(1)
                # Normalisation
                data = data.astype(np.float32)
                if data.max() > 0:
                    data = (data - data.min()) / (data.max() - data.min())
                    
                # Conversion PyTorch
                device = 'cuda' if torch.cuda.is_available() else 'cpu'
                img_tensor = torch.from_numpy(data).unsqueeze(0).unsqueeze(0).to(device)
                
                # Calcul Hessien via la classe importée
                steger = StegerHessian(σ=σ_val, device=device)
                # Correction : compute_hessian retourne 5 valeurs (ix, iy, ixx, ixy, iyy)
                ix, iy, ixx, ixy, iyy = steger.compute_hessian(img_tensor)
                λ1, λ2 = steger.compute_eigenvalues(ixx, ixy, iyy)
                
                # Affichage de lambda2 (valeur brute)
                result_np = λ2.squeeze().cpu().numpy()
                
                plt.figure(figsize=(12, 10))
                plt.imshow(result_np, cmap='RdBu_r')
                plt.colorbar(label='Eigenvalue λ2')
                plt.title(f"Hessian Eigenvalue λ2 (σ={σ_val}) : {target_file}")
                plt.show()
        except Exception as e:
             print(f"Erreur sur {target_file}: {e}")
    else:
        print(f"Fichier cible {target_file} introuvable dans {folder_path}")